In [2]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_frmri_articulatory_features

In [3]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[1])
token_df = dataset_df_to_token_df(df)

In [4]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.717,2.717,2.907,f,i,f,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
1,2.717,2.907,2.907,2.937,i,l,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
2,2.907,2.937,2.937,3.067,l,u,f_i_l,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
3,3.097,3.157,3.157,3.237,l,a,l,la,4,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
4,3.237,3.317,3.317,3.377,f,ʁ,f,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
...,...,...,...,...,...,...,...,...,...,...,...,...
22432,33.071,33.131,33.131,33.211,ʁ,e,p_ʁ,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22433,33.131,33.211,33.211,33.391,e,s,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22434,33.211,33.391,33.391,33.671,s,i,p_ʁ_e_s,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22435,33.391,33.671,33.671,33.801,i,z,p_ʁ_e_s_i,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(136, 136),
    latent_dim=64,
    input_channels=1,
    output_activation=None,
    stn_mode="shared",
    stn_transform="rigid",
).to(device)

ckpt = torch.load(
    "../encoder_training/ckpts/frmri/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()


ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/frmri/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [6]:
import cv2
with open("../encoder_training/scalers/frmri_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../encoder_training/scalers/frmri_ac.pkl", "rb") as f:
    scalers = pickle.load(f)

In [8]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    stats = recording_stats(art_f)
    articulatory_features = get_frmri_articulatory_features(art_f, start, end, stats=stats)
    if articulatory_features is None: continue
    articulatory_features = torch.from_numpy(articulatory_features).unsqueeze(0).unsqueeze(0).to(device)
    if articulatory_features.shape[3:]!=(136, 136): continue
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 30598/30598 [59:15<00:00,  8.61it/s]  


In [9]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (30485, 64) ac_flat: (30485, 64)


In [10]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [11]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [13]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=9.0552, val=-2.3423
   ✓ New best model
Epoch 2: train=-8.0407, val=-10.1419
   ✓ New best model
Epoch 3: train=-13.5311, val=-12.7662
   ✓ New best model
Epoch 4: train=-17.9051, val=-12.3364
   No improvement (1 epochs)
Epoch 5: train=-21.8111, val=-18.5058
   ✓ New best model
Epoch 6: train=-24.4526, val=-19.9741
   ✓ New best model
Epoch 7: train=-26.7231, val=-20.6798
   ✓ New best model
Epoch 8: train=-29.0924, val=-20.6323
   No improvement (1 epochs)
Epoch 9: train=-30.9453, val=-21.0204
   ✓ New best model
Epoch 10: train=-33.0538, val=-21.0103
   No improvement (1 epochs)
Epoch 11: train=-34.7187, val=-20.7087
   No improvement (2 epochs)
Epoch 12: train=-36.3584, val=-22.5810
   ✓ New best model
Epoch 13: train=-38.0522, val=-22.7267
   ✓ New best model
Epoch 14: train=-39.2429, val=-19.7081
   No improvement (1 epochs)
Epoch 15: train=-40.8334, val=-20.0738
   No improvement (2 epochs)
Epoch 16: train=-42.2150, val=-19.3199
   No improvement (3 epo

In [14]:
import pickle
with open("entropies/frmri_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [15]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=85.3239, val=84.2360
   ✓ New best model
Epoch 2: train=83.8552, val=83.7520
   ✓ New best model
Epoch 3: train=83.4620, val=83.5576
   ✓ New best model
Epoch 4: train=83.2804, val=83.4054
   ✓ New best model
Epoch 5: train=83.1630, val=83.4416
   No improvement (1 epochs)
Epoch 6: train=83.0241, val=83.2978
   ✓ New best model
Epoch 7: train=82.9242, val=83.3054
   No improvement (1 epochs)
Epoch 8: train=82.8431, val=83.1511
   ✓ New best model
Epoch 9: train=82.7259, val=83.0767
   ✓ New best model
Epoch 10: train=82.6504, val=83.0957
   No improvement (1 epochs)
Epoch 11: train=82.5496, val=83.1731
   No improvement (2 epochs)
Epoch 12: train=82.5083, val=83.0971
   No improvement (3 epochs)
Epoch 13: train=82.4247, val=82.9487
   ✓ New best model
Epoch 14: train=82.3617, val=83.0035
   No improvement (1 epochs)
Epoch 15: train=82.2955, val=82.9168
   ✓ New best model
Epoch 16: train=82.2422, val=82.8560
   ✓ New best model
Epoch 17: train=82.1811, val=82.

In [16]:
with open("entropies/frmri_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)